# Clearvale: download every subject's PsychoPy stim.csv files from Flywheel

Lists and downloads only the `*stim*.csv` files attached to each session's acquisitions (about 11 per session, ~250 KB). Nothing on Flywheel is changed. Files land in `sourcedata/stim/sub-<subject>/ses-<session>/<acquisition>/` next to this notebook -- the layout `code/make_events.py` and `code/audit_stim.py` read.

1. log in (hidden prompt), 2. inventory (no download), 3. download (skips files already here), 4. hand-off to the audit.

In [ ]:
import importlib.util
if importlib.util.find_spec("flywheel") is None:
    %pip install flywheel-sdk

In [ ]:
import getpass, os, re
from pathlib import Path
import pandas as pd
import flywheel

# the Clearvale BIDS folder: $CV_BIDS, else <repo>/data/cv-bids (run this notebook from examples/clearvale/notebooks)
OUT = Path(os.environ.get("CV_BIDS", Path.cwd().resolve().parents[2] / "data" / "cv-bids"))
PROJECT = "mbanich/CV"                   # group/project
STIM_PATTERN = r"stim.*\.csv$"           # file names to fetch (case-insensitive)
SUBJECTS = None                          # None = every subject; or a list of labels, e.g. ["090", "091"]
DOWNLOAD = True                          # False = inventory only

fw = flywheel.Client()
print("logged in as", fw.get_current_user().email)
project = fw.lookup(PROJECT)
print("project:", project.label)

## Inventory (no downloads)

In [ ]:
rows = []
subjects = [s for s in project.subjects() if SUBJECTS is None or s.label in SUBJECTS]
for i, subj in enumerate(sorted(subjects, key=lambda s: s.label)):
    for ses in subj.sessions():
        for acq in ses.acquisitions():
            for f in acq.reload().files or []:
                if re.search(STIM_PATTERN, f.name, re.I):
                    rows.append({"subject": subj.label, "session": ses.label, "acquisition": acq.label,
                                 "file": f.name, "size": f.size, "_acq": acq})
    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{len(subjects)} subjects listed, {len(rows)} stim files so far")
inv = pd.DataFrame(rows)
print(f"{len(inv)} stim files, {inv['size'].sum() / 1e6:.1f} MB, {inv['subject'].nunique()} subjects")
per = inv.groupby(["subject", "session"]).size().rename("n_files")
display(per.value_counts().rename("sessions").to_frame())
print("sessions without 11 stim files:"); display(per[per != 11].to_frame())

## Download

In [ ]:
if not DOWNLOAD:
    raise SystemExit("DOWNLOAD is False")
done = skipped = 0
for _, r in inv.iterrows():
    dest = OUT / "sourcedata" / "stim" / f"sub-{r['subject']}" / f"ses-{r['session']}" / r["acquisition"] / r["file"]
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size == r["size"]:
        skipped += 1
        continue
    r["_acq"].download_file(r["file"], str(dest))
    done += 1
inv.drop(columns="_acq").to_csv(OUT / "sourcedata" / "stim" / "stim_inventory.csv", index=False)
print(f"downloaded {done}, already here {skipped}; inventory -> {OUT / 'sourcedata/stim/stim_inventory.csv'}")

## Next

Run the audit (from the rsaworkflow repo root):

```
~/.venvs/rsaworkflow/bin/python data/cv-bids/code/audit_stim.py
```

It writes `audit/*.csv` and prints, per subject, WM images without a localizer match (exact name, then normalized name), label problems in the localizer and WM logs, and run / trial counts.